# 1. MNIST 真实数据生成 (宿主机运行)

本 notebook 在有 PyTorch 的宿主机上运行，完成：
1. 训练 784→128→10 MLP
2. PTQ 量化为 INT8（对齐 CIM 硬件）
3. Python bit-accurate 推理验证
4. 导出 hex 文件供 FPGA 加载

生成的 `mnist_real_data/` 目录传到 PYNQ 上，用 `02_pynq_mnist_test.ipynb` 验证。

In [1]:
import sys, os
# 确保 mnist_quantize.py 在路径中
sys.path.insert(0, '.')
from mnist_quantize import *

## 1.1 训练模型

In [2]:
SEED = 42          # 固定种子保证可复现；设为 None 则完全随机
EPOCHS = 10
NUM_TEST = 20      # 导出的测试图片数量
OUTPUT_DIR = 'mnist_real_data'
MODEL_PATH = 'mnist_mlp.pt'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

Device: cuda


In [3]:
# 训练（如果已有 model.pt 可跳过此 cell）
model, _, float_acc = train_model(EPOCHS, device=device, seed=SEED)
torch.save(model.state_dict(), MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')

Training MNIST MLP (784→128→10)
Seed: 42
  Epoch 1/10: loss=0.4098, acc=93.41%
  Epoch 2/10: loss=0.1932, acc=95.38%
  Epoch 3/10: loss=0.1402, acc=95.98%
  Epoch 4/10: loss=0.1091, acc=96.78%
  Epoch 5/10: loss=0.0880, acc=97.11%
  Epoch 6/10: loss=0.0728, acc=97.31%
  Epoch 7/10: loss=0.0616, acc=97.47%
  Epoch 8/10: loss=0.0515, acc=97.64%
  Epoch 9/10: loss=0.0433, acc=97.67%
  Epoch 10/10: loss=0.0370, acc=97.55%

Final float32 accuracy: 97.55%
Model saved to mnist_mlp.pt


In [4]:
# 或者加载已有模型（取消注释）
# model = MnistMLP()
# model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
# model = model.to(device).eval()
# print('Loaded pretrained model')

## 1.2 Post-Training Quantization

In [5]:
model = model.to(device).eval()

transform = transforms.Compose([
    transforms.ToTensor(),
])
test_set = datasets.MNIST('./data', train=False, download=True, transform=transform)
cal_loader = torch.utils.data.DataLoader(test_set, batch_size=1000)

qparams = full_ptq(model, cal_loader, device)

print(f"\nHW zero points: FC1={qparams['hw_zp1']}, FC2={qparams['hw_zp2']}")
print(f"Requant: FC1=({qparams['fc1_mult']}, {qparams['fc1_shift']})")  
print(f"         FC2=({qparams['fc2_mult']}, {qparams['fc2_shift']})")
print(f"Weight shapes: FC1={qparams['w1'].shape}, FC2={qparams['w2'].shape}")


Post-Training Quantization
  FC1 input: FIXED scale=1/255=0.003922, hw_zp=0
  FC1 weight: scale=0.005978, range=[-127, 87]
  FC2 weight: scale=0.006856, range=[-127, 90]
  Calibrating FC1 output range...
  FC1 output range: [0.0000, 13.7245]
  FC1 output / FC2 input scale: 0.108067, hw_zp=0
  FC1 requant: mult=14, shift=16
  FC2 requant: mult=140, shift=16

HW zero points: FC1=0, FC2=0
Requant: FC1=(14, 16)
         FC2=(140, 16)
Weight shapes: FC1=(128, 784), FC2=(10, 128)


## 1.3 INT8 准确率验证

In [6]:
transform_raw = transforms.ToTensor()
test_set_raw = datasets.MNIST('./data', train=False, download=True, transform=transform_raw)

correct_int8 = 0
correct_float = 0
total = len(test_set_raw)

for i in range(total):
    img_tensor, label = test_set_raw[i]
    img_float = img_tensor.numpy().flatten()
    img_uint8 = quantize_image(img_float, qparams['s_in1'], -qparams['hw_zp1'])
    
    pred_int8, _, _ = hw_infer_mlp(img_uint8, qparams)
    if pred_int8 == label: correct_int8 += 1
    
    with torch.no_grad():
        img_norm = test_set[i][0].unsqueeze(0).to(device)
        pred_float = model(img_norm).argmax(1).item()
    if pred_float == label: correct_float += 1

print(f'Float32 accuracy: {100.*correct_float/total:.2f}% ({correct_float}/{total})')
print(f'INT8 accuracy:    {100.*correct_int8/total:.2f}% ({correct_int8}/{total})')
print(f'Accuracy drop:    {100.*(correct_float-correct_int8)/total:.2f}%')

Float32 accuracy: 97.55% (9755/10000)
INT8 accuracy:    97.63% (9763/10000)
Accuracy drop:    -0.08%


## 1.4 导出 Hex 文件

In [7]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 权重 + bias + 量化参数（所有图片共享）
save_hex(weight_to_chunk_hex(qparams['w1']), f'{OUTPUT_DIR}/fc1_weight_tiles.hex')
save_hex(weight_to_chunk_hex(qparams['w2']), f'{OUTPUT_DIR}/fc2_weight_tiles.hex')
save_hex(bias_to_hex(qparams['b1']), f'{OUTPUT_DIR}/fc1_bias.hex')
save_hex(bias_to_hex(qparams['b2']), f'{OUTPUT_DIR}/fc2_bias.hex')
save_hex([
    f"{qparams['fc1_mult'] & 0xFFFFFFFF:08x}",
    f"{qparams['fc1_shift'] & 0xFFFFFFFF:08x}",
    f"{qparams['fc2_mult'] & 0xFFFFFFFF:08x}",
    f"{qparams['fc2_shift'] & 0xFFFFFFFF:08x}",
], f'{OUTPUT_DIR}/quant_params.hex')

# 零点信息（PYNQ 端需要）
save_hex([
    f"{qparams['hw_zp1'] & 0xFFFFFFFF:08x}",
    f"{qparams['hw_zp2'] & 0xFFFFFFFF:08x}",
], f'{OUTPUT_DIR}/zero_points.hex')

print(f'Weights/bias/quant saved to {OUTPUT_DIR}/')

Weights/bias/quant saved to mnist_real_data/


In [8]:
# 导出测试图片
img_dir = f'{OUTPUT_DIR}/test_images'
os.makedirs(img_dir, exist_ok=True)

n_export = min(NUM_TEST, len(test_set_raw))
export_correct = 0

for i in range(n_export):
    img_tensor, label = test_set_raw[i]
    img_float = img_tensor.numpy().flatten()
    img_uint8 = quantize_image(img_float, qparams['s_in1'], -qparams['hw_zp1'])
    
    pred, fc1_out, fc2_out = hw_infer_mlp(img_uint8, qparams)
    if pred == label: export_correct += 1
    
    prefix = f'img_{i:04d}'
    save_hex(input_to_hex(img_uint8), f'{img_dir}/{prefix}.hex')
    save_hex(int8_to_hex(fc1_out), f'{img_dir}/{prefix}_fc1.hex')
    save_hex(int8_to_hex(fc2_out), f'{img_dir}/{prefix}_fc2.hex')
    with open(f'{img_dir}/{prefix}_label.txt', 'w') as f: f.write(f'{label}\n')
    with open(f'{img_dir}/{prefix}_pred.txt', 'w') as f: f.write(f'{pred}\n')
    
    print(f'  [{i:04d}] label={label}, int8_pred={pred} {"✓" if pred==label else "✗"}')

print(f'\nExported {n_export} images, INT8 accuracy: {100.*export_correct/n_export:.1f}%')
print(f'\n>>> 把 {OUTPUT_DIR}/ 目录打包传到 PYNQ <<<')

  [0000] label=7, int8_pred=7 ✓
  [0001] label=2, int8_pred=2 ✓
  [0002] label=1, int8_pred=1 ✓
  [0003] label=0, int8_pred=0 ✓
  [0004] label=4, int8_pred=4 ✓
  [0005] label=1, int8_pred=1 ✓
  [0006] label=4, int8_pred=4 ✓
  [0007] label=9, int8_pred=9 ✓
  [0008] label=5, int8_pred=5 ✓
  [0009] label=9, int8_pred=9 ✓
  [0010] label=0, int8_pred=0 ✓
  [0011] label=6, int8_pred=6 ✓
  [0012] label=9, int8_pred=9 ✓
  [0013] label=0, int8_pred=0 ✓
  [0014] label=1, int8_pred=1 ✓
  [0015] label=5, int8_pred=5 ✓
  [0016] label=9, int8_pred=9 ✓
  [0017] label=7, int8_pred=7 ✓
  [0018] label=3, int8_pred=3 ✓
  [0019] label=4, int8_pred=4 ✓

Exported 20 images, INT8 accuracy: 100.0%

>>> 把 mnist_real_data/ 目录打包传到 PYNQ <<<


In [9]:
# 打包为 zip 方便传输
import shutil
shutil.make_archive(OUTPUT_DIR, 'zip', '.', OUTPUT_DIR)
print(f'Created {OUTPUT_DIR}.zip — 传到 PYNQ Jupyter 同目录下')

Created mnist_real_data.zip — 传到 PYNQ Jupyter 同目录下
